# Ablation 5: Minimal Diagnostic

**Purpose**: Check the label smoothing assumptions in 2-3 minutes.

**Tests**:
1. Package installation (ultralytics, numpy)
2. Dataset access (baseline-outputs)
3. YOLO config creation
4. Label smoothing parameter acceptance
5. Model initialization with label_smoothing=0.1

**Runtime**: 2-3 minutes (CPU only)
**Success**: all 5 tests pass. Next: smoke test.
**Failure**: any test fails. Record the exact error.

## Test 1: Package Installation

ultralytics and numpy install correctly.

In [ ]:
# Install packages
!pip install -q 'ultralytics>=8.3.40' 'numpy<2.0'

# Validate installation
import importlib.util
assert importlib.util.find_spec('ultralytics') is not None, "ultralytics not installed"
assert importlib.util.find_spec('numpy') is not None, "numpy not installed"

import ultralytics
import numpy as np
print(f'Ultralytics: {ultralytics.__version__}')
print(f'NumPy: {np.__version__}')
print('\nOK TEST 1 PASSED: All packages installed successfully')

## Test 2: Dataset Access

Validate baseline-outputs dataset exists and has expected structure.

In [ ]:
from pathlib import Path
import os

# Check dataset exists
dataset_path = Path('/kaggle/input/baseline-outputs/data_wound_seg')
assert dataset_path.exists(), f"Dataset not found: {dataset_path}"

# Check subdirectories
train_imgs = dataset_path / 'train/images'
train_labels = dataset_path / 'train/labels'
test_imgs = dataset_path / 'test/images'
test_labels = dataset_path / 'test/labels'

assert train_imgs.exists(), f"Train images not found: {train_imgs}"
assert train_labels.exists(), f"Train labels not found: {train_labels}"
assert test_imgs.exists(), f"Test images not found: {test_imgs}"
assert test_labels.exists(), f"Test labels not found: {test_labels}"

# Count files
n_train_imgs = len(list(train_imgs.glob('*.png')))
n_train_labels = len(list(train_labels.glob('*.txt')))
n_test_imgs = len(list(test_imgs.glob('*.png')))
n_test_labels = len(list(test_labels.glob('*.txt')))

print(f'Train: {n_train_imgs} images, {n_train_labels} labels')
print(f'Test: {n_test_imgs} images, {n_test_labels} labels')

assert n_train_imgs > 0, "No training images found"
assert n_test_imgs > 0, "No test images found"

print('\nOK TEST 2 PASSED: Dataset accessible with expected structure')

## Test 3: YOLO Config Creation

The YOLO configuration file can be created.

In [ ]:
import yaml

# Create config
config_content = {
    'path': str(dataset_path),
    'train': 'train',
    'val': 'test',
    'names': {0: 'wound'}
}

config_path = Path('/kaggle/working/wound_dataset.yaml')
with open(config_path, 'w') as f:
    yaml.dump(config_content, f)

# Verify config
assert config_path.exists(), "Config file not created"

with open(config_path) as f:
    loaded_config = yaml.safe_load(f)

assert loaded_config['path'] == str(dataset_path), "Config path mismatch"
assert loaded_config['names'][0] == 'wound', "Config names mismatch"

print(f'Config created: {config_path}')
print(f'Contents:')
print(yaml.dump(loaded_config, default_flow_style=False))
print('\nOK TEST 3 PASSED: YOLO config created successfully')

## Test 4: Label Smoothing Parameter Acceptance

ultralytics accepts the `label_smoothing=0.1` parameter without errors.

**Check**: whether the YOLO constructor accepts this parameter.

In [ ]:
from ultralytics import YOLO

# Create model instance
model = YOLO("yolov8n-seg.pt")
print(f'Model created: {type(model)}')
print(f'Model task: {model.task}')

# Test parameter acceptance by inspecting training args
# We don't train, just check if parameter would be accepted
try:
    # Check if label_smoothing is a valid training argument
    from ultralytics.cfg import get_cfg
    from ultralytics.utils import DEFAULT_CFG_DICT

    # Create config with label_smoothing
    cfg = get_cfg(DEFAULT_CFG_DICT)
    cfg.label_smoothing = 0.1

    print(f'\nLabel smoothing parameter set: {cfg.label_smoothing}')
    print('Parameter type:', type(cfg.label_smoothing))

    # Note: ultralytics >=8.3.40 has deprecated label_smoothing but still accepts it
    print('\nOK TEST 4 PASSED: label_smoothing parameter accepted')
except Exception as e:
    print(f'\nFAIL TEST 4 FAILED: {e}')
    raise

## Test 5: Model Initialization

The model runs inference (no training needed).

In [ ]:
# Get first test image
test_images = list(test_imgs.glob('*.png'))
assert len(test_images) > 0, "No test images found"

test_img = test_images[0]
print(f'Testing on: {test_img.name}')

# Run inference
results = model(str(test_img), verbose=False)

# Inspect result
print(f'\nInference result:')
print(f'  Type: {type(results[0])}')
print(f'  Has masks: {results[0].masks is not None}')

if results[0].masks is not None:
    print(f'  Number of masks: {len(results[0].masks)}')
    print('\nOK TEST 5 PASSED: Model inference works')
else:
    print('  No masks detected (model works, just no detections)')
    print('\nOK TEST 5 PASSED: Model inference works (no detections)')

## Test Summary

In [ ]:
import json
from datetime import datetime

test_summary = {
    'notebook': 'ablation_5_minimal_diagnostic',
    'timestamp': datetime.now().isoformat(),
    'tests': [
        {'test': 1, 'name': 'Package Installation', 'status': 'Check output above'},
        {'test': 2, 'name': 'Dataset Access', 'status': 'Check output above'},
        {'test': 3, 'name': 'YOLO Config Creation', 'status': 'Check output above'},
        {'test': 4, 'name': 'Label Smoothing Parameter', 'status': 'Check output above'},
        {'test': 5, 'name': 'Model Initialization', 'status': 'Check output above'}
    ],
    'runtime_target': '2-3 minutes',
    'next_step': 'If all tests pass → deploy smoke test (1 epoch training)',
    'on_failure': 'Report exact error, fix root cause'
}

with open('/kaggle/working/test_summary.json', 'w') as f:
    json.dump(test_summary, f, indent=2)

print('\n' + '='*60)
print('MINIMAL DIAGNOSTIC COMPLETE')
print('='*60)
print('\nReview test results above:')
print('  - All tests should show OK PASSED')
print('  - Any FAIL FAILED indicates assumption violation')
print('\nNext steps:')
print('  - If ALL pass: Deploy smoke test (1 epoch training)')
print('  - If ANY fail: Debug root cause before proceeding')
print('\nOutputs:')
print('  - test_summary.json')